# Neighborhood Action Dashboard — Notebook

**FIAP · Ciência da Computação · Global Solution 2026.2 · CitizenTech**

Este notebook reúne o mesmo código do arquivo `src/sistema.py`, dividido em etapas com explicações e demonstrações prontas. O sistema simula um painel comunitário: registra ocorrências fictícias de bairro (iluminação, acessibilidade, limpeza e manutenção), consulta e filtra os registros, atualiza a situação de cada demanda com histórico e calcula indicadores.

**Como usar:** execute as células em ordem (*Run All*). As demonstrações das seções 4 e 5 cadastram e alteram a ocorrência 13, e o painel da seção 8 já reflete essas mudanças. Na última seção há o menu interativo, igual ao do terminal.

> Todos os dados são **fictícios**. Não há dados pessoais, integração com órgãos públicos ou promessa de atendimento.

**Conceitos aplicados:** listas, filas, pilhas, dicionários, matriz (lista de listas), hierarquia aninhada, `if/elif/else`, operadores `and`/`or`/`not`, laços `for`/`while`, funções, validação de entrada e cálculo de indicadores.


## 1. Dados e estruturas

Cada ocorrência tem seis campos: `identificador`, `categoria`, `descricao`, `local`, `data` e `situacao`. São **obrigatórios** `categoria`, `descricao`, `local` e `data`; o `identificador` e a `situacao` inicial (`Aberta`) são gerados pelo sistema.

**Valores padronizados**

| Situação | Significado |
| --- | --- |
| `Aberta` | Demanda registrada, ainda não examinada |
| `Em analise` | Demanda já examinada, aguardando solução |
| `Concluida` | Demanda resolvida |

| Estrutura | Papel no sistema |
| --- | --- |
| **Lista** `ocorrencias` | Guarda todos os registros e permite percorrê-los |
| **Fila** `fila_analise` (FIFO) | Demandas ativas, por ordem de registro |
| **Pilha** `pilha_historico` (LIFO) | Atualizações de status; o topo é o evento mais recente |
| **Dicionário** `dicionario_ocorrencias` | Acesso direto por identificador |
| **Hierarquia** `hierarquia_bairros` | Bairro → categoria → lista de IDs |
| **Matriz** (em `calcular_indicadores`) | Contagem por categoria × situação |

O registro **12** tem a categoria vazia de propósito, para testar a validação. O histórico inicial tem 10 eventos.

In [1]:
"""
Neighborhood Action Dashboard
FIAP - Ciencia da Computacao - Global Solution 2026.2 - CitizenTech

Prototipo de terminal para registrar, consultar, atualizar e resumir
ocorrencias comunitarias FICTICIAS (sem dados pessoais).
Execucao: python src/sistema.py
"""

# Valores padronizados
CATEGORIAS = ["iluminacao", "acessibilidade", "limpeza", "manutencao"]
SITUACOES = ["Aberta", "Em analise", "Concluida"]

# "Hoje" simulado, usado para calcular o tempo de acompanhamento
DATA_REFERENCIA = "15/11/2026"

# Fila (FIFO): demandas ativas aguardando analise, por ordem de registro
fila_analise = []

# Pilha (LIFO): ultimas atualizacoes de status (o topo e o ultimo elemento)
pilha_historico = []

# Dicionario: identificador -> ocorrencia (acesso direto O(1))
dicionario_ocorrencias = {}

# Hierarquia: bairro -> categoria -> lista de identificadores
hierarquia_bairros = {}

# Lista principal: percorrida em listagens, filtros e indicadores
ocorrencias = [
    {"identificador": 1, "categoria": "iluminacao", "descricao": "Poste apagado na rua principal", "local": "Bairro Aurora", "data": "01/11/2026", "situacao": "Aberta"},
    {"identificador": 2, "categoria": "iluminacao", "descricao": "Falta de luz na passarela", "local": "Bairro Horizonte", "data": "03/11/2026", "situacao": "Concluida"},
    {"identificador": 3, "categoria": "acessibilidade", "descricao": "Piso tatil danificado", "local": "Bairro Aurora", "data": "05/11/2026", "situacao": "Em analise"},
    {"identificador": 4, "categoria": "limpeza", "descricao": "Lixo acumulado na esquina", "local": "Bairro Horizonte", "data": "07/11/2026", "situacao": "Concluida"},
    {"identificador": 5, "categoria": "limpeza", "descricao": "Entulho na calcada", "local": "Praca Central", "data": "09/11/2026", "situacao": "Em analise"},
    {"identificador": 6, "categoria": "manutencao", "descricao": "Portao do parque enferrujado", "local": "Bairro Horizonte", "data": "11/11/2026", "situacao": "Em analise"},
    {"identificador": 7, "categoria": "manutencao", "descricao": "Quadra com buracos", "local": "Centro Comunitario", "data": "12/11/2026", "situacao": "Concluida"},
    {"identificador": 8, "categoria": "manutencao", "descricao": "Banco quebrado na praca", "local": "Praca Central", "data": "10/11/2026", "situacao": "Aberta"},
    {"identificador": 9, "categoria": "limpeza", "descricao": "Mato alto no terreno publico", "local": "Bairro Aurora", "data": "08/11/2026", "situacao": "Aberta"},
    {"identificador": 10, "categoria": "acessibilidade", "descricao": "Falta de vaga para cadeirantes", "local": "Centro Comunitario", "data": "06/11/2026", "situacao": "Aberta"},
    {"identificador": 11, "categoria": "iluminacao", "descricao": "Lampada piscando na praca", "local": "Praca Central", "data": "04/11/2026", "situacao": "Aberta"},
    # Registro inconsistente (categoria ausente), usado para testar a validacao logica
    {"identificador": 12, "categoria": "", "descricao": "Rampa sem acesso na calcada", "local": "Praca Central", "data": "02/11/2026", "situacao": "Em analise"},
]

# Historico simulado inicial (10 eventos, do mais antigo ao mais recente)
HISTORICO_INICIAL = [
    "[04/11/2026] ID 2: 'Aberta' -> 'Em analise'",
    "[05/11/2026] ID 2: 'Em analise' -> 'Concluida'",
    "[06/11/2026] ID 3: 'Aberta' -> 'Em analise'",
    "[08/11/2026] ID 4: 'Aberta' -> 'Em analise'",
    "[09/11/2026] ID 4: 'Em analise' -> 'Concluida'",
    "[10/11/2026] ID 5: 'Aberta' -> 'Em analise'",
    "[12/11/2026] ID 6: 'Aberta' -> 'Em analise'",
    "[13/11/2026] ID 7: 'Aberta' -> 'Em analise'",
    "[14/11/2026] ID 7: 'Em analise' -> 'Concluida'",
    "[14/11/2026] ID 12: 'Aberta' -> 'Em analise'",
]

## 2. Inicialização das estruturas auxiliares

Percorremos a lista principal uma vez para preencher o dicionário, a fila (só demandas `Aberta` ou `Em analise`) e a hierarquia. Registros sem categoria válida vão para o grupo `Indefinida`. Depois empilhamos o histórico inicial.

In [2]:
def indexar_hierarquia(oc):
    """Insere a ocorrencia na hierarquia bairro -> categoria -> ids."""
    bairro = oc["local"]
    cat = oc["categoria"] if oc["categoria"] in CATEGORIAS else "Indefinida"
    if bairro not in hierarquia_bairros:
        hierarquia_bairros[bairro] = {}
    if cat not in hierarquia_bairros[bairro]:
        hierarquia_bairros[bairro][cat] = []
    hierarquia_bairros[bairro][cat].append(oc["identificador"])


for oc in ocorrencias:
    dicionario_ocorrencias[oc["identificador"]] = oc
    if oc["situacao"] in ["Aberta", "Em analise"]:
        fila_analise.append(oc["identificador"])
    indexar_hierarquia(oc)

for evento in HISTORICO_INICIAL:
    pilha_historico.append(evento)

## 3. Utilitários de data

As datas são texto no formato `DD/MM/AAAA`. Para medir o tempo de acompanhamento, `data_para_dias()` converte a data em dias corridos **aproximados** (meses de 30 dias), o que basta para uma simulação.

In [3]:
def data_valida(data):
    """True se a data estiver no formato DD/MM/AAAA com valores plausiveis."""
    partes = data.split("/")
    if len(partes) != 3:
        return False
    for p in partes:
        if not p.isdigit():
            return False
    dia, mes, ano = int(partes[0]), int(partes[1]), int(partes[2])
    return len(partes[2]) == 4 and 1 <= dia <= 31 and 1 <= mes <= 12


def data_para_dias(data):
    """Converte DD/MM/AAAA em dias corridos aproximados (meses de 30 dias)."""
    partes = data.split("/")
    return int(partes[2]) * 360 + int(partes[1]) * 30 + int(partes[0])

## 4. Validação e cadastro

`validar_ocorrencia()` só devolve `True` quando todos os campos estão preenchidos, a categoria é conhecida e a data tem formato válido. A expressão central usa `not` e `or`:

```python
if not categoria or not descricao or not local or not data:
```

Ela é verdadeira (e bloqueia o cadastro) quando **pelo menos um** campo obrigatório está vazio. Quando o cadastro passa, o novo registro entra na lista, no dicionário, no fim da fila e na hierarquia.

In [4]:
def validar_ocorrencia(categoria, descricao, local, data):
    """Retorna (ok, mensagem). Nao imprime nada."""
    # NOT + OR: reprova se QUALQUER campo obrigatorio estiver vazio
    if not categoria or not descricao or not local or not data:
        return False, "Por favor, complete todo o formulario!"
    if categoria not in CATEGORIAS:
        return False, "Categoria invalida! Use: " + ", ".join(CATEGORIAS) + "."
    if not data_valida(data):
        return False, "Data invalida! Use o formato DD/MM/AAAA."
    return True, "Dados validos."


def cadastrar_ocorrencia(categoria, descricao, local, data):
    """Valida e cadastra. Retorna (ok, mensagem, identificador)."""
    ok, mensagem = validar_ocorrencia(categoria, descricao, local, data)
    if not ok:
        return False, mensagem, None

    identificador = len(ocorrencias) + 1
    nova_oc = {
        "identificador": identificador,
        "categoria": categoria,
        "descricao": descricao,
        "local": local,
        "data": data,
        "situacao": "Aberta",
    }
    ocorrencias.append(nova_oc)                    # lista
    dicionario_ocorrencias[identificador] = nova_oc  # dicionario
    fila_analise.append(identificador)             # fila (entra no fim)
    indexar_hierarquia(nova_oc)                    # hierarquia
    return True, f"Ocorrencia ID {identificador} registrada com sucesso!", identificador

**Demonstração:** três tentativas inválidas e um cadastro válido (equivalente ao exemplo do enunciado: iluminação, Praça Central).

In [5]:
print("1) Sem categoria:   ", cadastrar_ocorrencia("", "Poste queimado", "Praca Central", "14/11/2026")[:2])
print("2) Categoria falsa:  ", cadastrar_ocorrencia("pintura", "Muro sujo", "Praca Central", "14/11/2026")[:2])
print("3) Data invalida:    ", cadastrar_ocorrencia("iluminacao", "Poste queimado", "Praca Central", "31-11-2026")[:2])
print("4) Cadastro valido:  ", cadastrar_ocorrencia("iluminacao", "Poste queimado na quadra", "Praca Central", "14/11/2026"))

1) Sem categoria:    (False, 'Por favor, complete todo o formulario!')
2) Categoria falsa:   (False, 'Categoria invalida! Use: iluminacao, acessibilidade, limpeza, manutencao.')
3) Data invalida:     (False, 'Data invalida! Use o formato DD/MM/AAAA.')
4) Cadastro valido:   (True, 'Ocorrencia ID 13 registrada com sucesso!', 13)


## 5. Atualização de situação e histórico

`atualizar_situacao()` recusa identificador inexistente, situação desconhecida e mudança para a mesma situação. Quando aceita, ela faz duas coisas: empilha o evento no histórico e ajusta a fila (concluída sai; aberta ou em análise entra se ainda não estiver).

```python
if nova_situacao == "Concluida" and id_busca in fila_analise:
    fila_analise.remove(id_busca)
elif nova_situacao in ["Aberta", "Em analise"] and id_busca not in fila_analise:
    fila_analise.append(id_busca)
```

In [6]:
def registrar_historico(evento):
    """Empilha um evento (LIFO)."""
    pilha_historico.append(evento)


def atualizar_situacao(id_busca, nova_situacao):
    """Atualiza a situacao e registra o historico. Retorna (ok, mensagem)."""
    if id_busca not in dicionario_ocorrencias:
        return False, "Ocorrencia nao encontrada no sistema."
    if nova_situacao not in SITUACOES:
        return False, "Situacao invalida! Escolha entre: Aberta, Em analise ou Concluida."

    oc = dicionario_ocorrencias[id_busca]
    status_antigo = oc["situacao"]
    if status_antigo == nova_situacao:
        return False, "A nova situacao e identica a atual. Nenhuma alteracao feita."

    oc["situacao"] = nova_situacao

    # 1. Pilha de historico
    evento = f"[{DATA_REFERENCIA}] ID {id_busca}: '{status_antigo}' -> '{nova_situacao}'"
    registrar_historico(evento)

    # 2. Fila de analise: concluida sai; reaberta/em analise volta se nao estiver
    if nova_situacao == "Concluida" and id_busca in fila_analise:
        fila_analise.remove(id_busca)
    elif nova_situacao in ["Aberta", "Em analise"] and id_busca not in fila_analise:
        fila_analise.append(id_busca)

    return True, f"Situacao atualizada! Historico: {evento}"

**Demonstração:** alterando a ocorrência 13 e testando os erros.

In [7]:
print(atualizar_situacao(13, "Em analise"))
print(atualizar_situacao(13, "Em analise"))   # igual a atual
print(atualizar_situacao(13, "Fechada"))      # situacao desconhecida
print(atualizar_situacao(99, "Concluida"))    # id inexistente

(True, "Situacao atualizada! Historico: [15/11/2026] ID 13: 'Aberta' -> 'Em analise'")
(False, 'A nova situacao e identica a atual. Nenhuma alteracao feita.')
(False, 'Situacao invalida! Escolha entre: Aberta, Em analise ou Concluida.')
(False, 'Ocorrencia nao encontrada no sistema.')


## 6. Consultas e filtros

A busca por identificador usa o dicionário (`id in dicionario_ocorrencias`). O filtro aceita categoria, situação ou os dois. Com os dois ativos, o operador `and` exige que o registro atenda às duas condições.

In [8]:
def filtrar(categoria=None, situacao=None):
    """Filtra por categoria e/ou situacao (None = sem filtro neste campo)."""
    resultado = []
    for oc in ocorrencias:
        passa_categoria = categoria is None or oc["categoria"] == categoria
        passa_situacao = situacao is None or oc["situacao"] == situacao
        # AND: com os dois filtros ativos, o registro precisa atender aos dois
        if passa_categoria and passa_situacao:
            resultado.append(oc)
    return resultado

In [9]:
print("Iluminacao E Aberta:")
for oc in filtrar(categoria="iluminacao", situacao="Aberta"):
    print(" ", oc["identificador"], oc["descricao"], "|", oc["local"])

print("\nTodas 'Em analise':", [oc["identificador"] for oc in filtrar(situacao="Em analise")])
print("Busca por codigo 12:", dicionario_ocorrencias.get(12))
print("Busca por codigo 99:", dicionario_ocorrencias.get(99, "nao encontrado"))

Iluminacao E Aberta:
  1 Poste apagado na rua principal | Bairro Aurora
  11 Lampada piscando na praca | Praca Central

Todas 'Em analise': [3, 5, 6, 12, 13]
Busca por codigo 12: {'identificador': 12, 'categoria': '', 'descricao': 'Rampa sem acesso na calcada', 'local': 'Praca Central', 'data': '02/11/2026', 'situacao': 'Em analise'}
Busca por codigo 99: nao encontrado


## 7. Indicadores

`calcular_indicadores()` percorre a lista uma vez e produz:

- **Total e proporção por situação**: `quantidade / total × 100`
- **Total e proporção por categoria**: `quantidade / total × 100` (categoria inválida conta como `Indefinida`)
- **Matriz categoria × situação**: lista de listas, uma linha por categoria e uma coluna por situação
- **Taxa de resolução**: `concluídas / total × 100`
- **Pendências**: quantidade de `Aberta` + `Em analise`
- **Tempo médio simulado das pendências**: `soma(dias desde a abertura) / pendências`, com `DATA_REFERENCIA` como "hoje" simulado

In [10]:
def calcular_indicadores():
    """Calcula todos os indicadores e devolve um dicionario."""
    linhas = CATEGORIAS + ["Indefinida"]
    cont_situacao = {s: 0 for s in SITUACOES}
    cont_categoria = {c: 0 for c in linhas}
    matriz = [[0, 0, 0] for _ in linhas]   # linhas = categorias, colunas = situacoes
    inconsistentes = []
    pendencias_local = {}
    soma_dias = 0
    qtd_pendentes = 0

    for oc in ocorrencias:
        sit = oc["situacao"]
        if oc["categoria"] in CATEGORIAS:
            cat = oc["categoria"]
        else:
            cat = "Indefinida"
            inconsistentes.append(oc["identificador"])

        cont_categoria[cat] += 1
        if sit in SITUACOES:
            cont_situacao[sit] += 1
            matriz[linhas.index(cat)][SITUACOES.index(sit)] += 1

            # Pendente = ainda nao concluida
            if sit != "Concluida":
                qtd_pendentes += 1
                local = oc["local"]
                pendencias_local[local] = pendencias_local.get(local, 0) + 1
                if data_valida(oc["data"]):
                    soma_dias += data_para_dias(DATA_REFERENCIA) - data_para_dias(oc["data"])

    total = len(ocorrencias)
    return {
        "total": total,
        "linhas": linhas,
        "cont_situacao": cont_situacao,
        "cont_categoria": cont_categoria,
        "matriz": matriz,
        "inconsistentes": inconsistentes,
        "pendentes": qtd_pendentes,
        "pendencias_local": pendencias_local,
        "taxa_resolucao": (cont_situacao["Concluida"] / total * 100) if total > 0 else 0.0,
        "tempo_medio_pendentes": (soma_dias / qtd_pendentes) if qtd_pendentes > 0 else 0.0,
    }

## 8. Interface de terminal

Funções que apenas mostram ou leem informações. Elas chamam a lógica das seções anteriores.

In [11]:
def texto_categoria(oc):
    return oc["categoria"] if oc["categoria"] else "(ausente)"


def mostrar_todas_ocorrencias():
    if not ocorrencias:
        print("Nenhuma ocorrencia cadastrada.")
        return

    print("\n--- Listagem Geral de Ocorrencias ---")
    print(f"{'ID':<4}| {'Categoria':<15}| {'Local':<19}| {'Status':<11}| Data")
    print("-" * 66)
    for oc in ocorrencias:
        print(f"{oc['identificador']:<4}| {texto_categoria(oc):<15}| {oc['local']:<19}| {oc['situacao']:<11}| {oc['data']}")


def ler_inteiro(mensagem):
    while True:
        entrada = input(mensagem).strip()
        if entrada.isdigit():
            return int(entrada)
        print("Digite apenas numeros validos!")


def mostrar_detalhes(oc):
    for chave, valor in oc.items():
        print(f"{chave}: {valor}")


def encontrar_ocorrencia():
    id_busca = ler_inteiro("Digite o numero do identificador da ocorrencia: ")

    # Busca O(1) via dicionario
    if id_busca in dicionario_ocorrencias:
        print("\n--- Ocorrencia Encontrada ---")
        mostrar_detalhes(dicionario_ocorrencias[id_busca])
    else:
        print("Ocorrencia nao encontrada no sistema.")

In [12]:
def registrar_ocorrencia():
    while True:
        print("\n---- Cadastro de Ocorrencia ----")
        print("(Nao informe nomes, telefones ou enderecos exatos: use apenas dados genericos.)")
        while True:
            categoria = input("Categoria (iluminacao, acessibilidade, limpeza, manutencao): ").strip().lower()
            if categoria in CATEGORIAS:
                break
            print("Opcao Invalida! Digite uma das categorias validas.")

        descricao = input("Descricao: ").strip()
        local = input("Local generico: ").strip()
        data = input("Data simulada (DD/MM/AAAA): ").strip()

        ok, mensagem, _ = cadastrar_ocorrencia(categoria, descricao, local, data)
        print(mensagem)

        continuar = input("Gostaria de cadastrar outra ocorrencia (y/n)? ").strip().lower()
        # AND: so encerra se a resposta nao for "y" E nao for "s"
        if continuar != "y" and continuar != "s":
            break


def imprimir_resultado_filtro(lista):
    if not lista:
        print("Nenhuma ocorrencia encontrada para este filtro.")
        return
    for oc in lista:
        print(f"ID: {oc['identificador']} | Categoria: {texto_categoria(oc)} | "
              f"Local: {oc['local']} | Status: {oc['situacao']} | {oc['descricao']}")
    print(f"Total encontrado: {len(lista)}")


def filtrar_ocorrencias():
    print("\n--- Filtrar Ocorrencias ---")
    print("1 - Filtrar por Categoria")
    print("2 - Filtrar por Status")
    print("3 - Filtrar por Categoria E Status")
    opcao_filtro = input("Escolha o tipo de filtro: ").strip()

    if opcao_filtro == "1":
        cat = input("Categoria (iluminacao, acessibilidade, limpeza ou manutencao): ").strip().lower()
        if cat not in CATEGORIAS:
            print("Categoria invalida.")
        else:
            imprimir_resultado_filtro(filtrar(categoria=cat))
    elif opcao_filtro == "2":
        status = input("Situacao (Aberta, Em analise, Concluida): ").strip().capitalize()
        if status not in SITUACOES:
            print("Situacao invalida.")
        else:
            imprimir_resultado_filtro(filtrar(situacao=status))
    elif opcao_filtro == "3":
        cat = input("Categoria (iluminacao, acessibilidade, limpeza ou manutencao): ").strip().lower()
        status = input("Situacao (Aberta, Em analise, Concluida): ").strip().capitalize()
        if cat not in CATEGORIAS or status not in SITUACOES:   # NOT/OR: algum valor invalido
            print("Categoria ou situacao invalida.")
        else:
            imprimir_resultado_filtro(filtrar(categoria=cat, situacao=status))
    else:
        print("Opcao de filtro invalida.")


def atualizar_situacao_ocorrencia():
    id_busca = ler_inteiro("Numero do Identificador da Ocorrencia: ")

    if id_busca not in dicionario_ocorrencias:
        print("Ocorrencia nao encontrada no sistema.")
        return

    print(f"Ocorrencia {id_busca} encontrada. Status atual: {dicionario_ocorrencias[id_busca]['situacao']}")
    nova = input("Nova situacao (Aberta / Em analise / Concluida): ").strip().capitalize()
    ok, mensagem = atualizar_situacao(id_busca, nova)
    print(mensagem)

In [13]:
def ver_resumo_indicadores():
    if len(ocorrencias) == 0:
        print("Nenhuma ocorrencia cadastrada.")
        return

    ind = calcular_indicadores()
    total = ind["total"]

    for id_inc in ind["inconsistentes"]:
        print(f"[ALERTA DE VALIDACAO] Ocorrencia ID {id_inc} possui Categoria ausente/inconsistente!")

    print("\n==========================================")
    print("      PAINEL DE INDICADORES (DASHBOARD)   ")
    print("==========================================")
    print(f"Total de ocorrencias cadastradas: {total}")
    print(f"Registros com dados inconsistentes: {len(ind['inconsistentes'])}\n")

    print("--- Indicadores por Situacao ---")
    print("Regra: proporcao = quantidade da situacao / total x 100")
    for sit, qtd in ind["cont_situacao"].items():
        print(f"- {sit}: {qtd} ({qtd / total * 100:.1f}%)")

    print("\n--- Indicadores por Categoria ---")
    print("Regra: proporcao = quantidade da categoria / total x 100")
    for cat, qtd in ind["cont_categoria"].items():
        print(f"- {cat}: {qtd} ({qtd / total * 100:.1f}%)")

    print("\n--- Matriz Sintese (Categoria x Situacao) ---")
    print(f"{'Categoria':<15} | {'Aberta':<8} | {'Em analise':<10} | {'Concluida':<9}")
    print("-" * 50)
    for i, cat in enumerate(ind["linhas"]):
        linha = ind["matriz"][i]
        print(f"{cat:<15} | {linha[0]:<8} | {linha[1]:<10} | {linha[2]:<9}")

    print("\n--- Indicadores Operacionais Extras ---")
    print(f"- Taxa de resolucao = concluidas / total x 100 = {ind['taxa_resolucao']:.1f}%")
    print(f"- Pendencias (Aberta + Em analise): {ind['pendentes']}")
    print(f"- Demandas ativas na fila de analise: {len(fila_analise)}")
    print(f"- Tempo medio simulado das pendencias = soma(dias desde a abertura) / pendencias")
    print(f"  (data de referencia {DATA_REFERENCIA}): {ind['tempo_medio_pendentes']:.1f} dias")
    print(f"- Atualizacoes no historico (pilha): {len(pilha_historico)}")

    print("\n--- Leitura dos resultados ---")
    maior = max(ind["cont_categoria"][c] for c in CATEGORIAS)
    lideres = [c for c in CATEGORIAS if ind["cont_categoria"][c] == maior]
    if len(lideres) > 1:
        print(f"- Categorias com mais registros (empate com {maior} cada): {', '.join(lideres)}.")
    else:
        print(f"- Categoria com mais registros: {lideres[0]} ({maior}).")
    print(f"- {ind['pendentes']} de {total} demandas ainda estao pendentes ({ind['pendentes'] / total * 100:.1f}%).")
    if ind["pendencias_local"]:
        pior_local = max(ind["pendencias_local"], key=ind["pendencias_local"].get)
        print(f"- Local com mais pendencias: {pior_local} ({ind['pendencias_local'][pior_local]}).")
    print("==========================================\n")


def mostrar_historico(ultimos=5):
    print(f"\n--- Ultimas {ultimos} atualizacoes (pilha, topo primeiro) ---")
    if not pilha_historico:
        print("Nenhuma atualizacao registrada.")
        return
    for evento in pilha_historico[::-1][:ultimos]:
        print(evento)
    print(f"(Total no historico: {len(pilha_historico)})")


def mostrar_fila():
    print("\n--- Fila de analise (FIFO: o primeiro da lista e o mais antigo) ---")
    if not fila_analise:
        print("Fila vazia.")
        return
    for posicao, id_oc in enumerate(fila_analise, start=1):
        oc = dicionario_ocorrencias[id_oc]
        print(f"{posicao:>2}. ID {id_oc} | {texto_categoria(oc)} | {oc['local']} | {oc['situacao']}")


def mostrar_hierarquia():
    print("\n--- Hierarquia: Bairro -> Categoria -> Ocorrencias ---")
    for bairro, categorias in hierarquia_bairros.items():
        print(bairro)
        for cat, ids in categorias.items():
            print(f"  +-- {cat}: IDs {ids}")

**Demonstração das estruturas:** listagem, hierarquia, fila e pilha de histórico.

In [14]:
mostrar_todas_ocorrencias()
mostrar_hierarquia()
mostrar_fila()
mostrar_historico()


--- Listagem Geral de Ocorrencias ---
ID  | Categoria      | Local              | Status     | Data
------------------------------------------------------------------
1   | iluminacao     | Bairro Aurora      | Aberta     | 01/11/2026
2   | iluminacao     | Bairro Horizonte   | Concluida  | 03/11/2026
3   | acessibilidade | Bairro Aurora      | Em analise | 05/11/2026
4   | limpeza        | Bairro Horizonte   | Concluida  | 07/11/2026
5   | limpeza        | Praca Central      | Em analise | 09/11/2026
6   | manutencao     | Bairro Horizonte   | Em analise | 11/11/2026
7   | manutencao     | Centro Comunitario | Concluida  | 12/11/2026
8   | manutencao     | Praca Central      | Aberta     | 10/11/2026
9   | limpeza        | Bairro Aurora      | Aberta     | 08/11/2026
10  | acessibilidade | Centro Comunitario | Aberta     | 06/11/2026
11  | iluminacao     | Praca Central      | Aberta     | 04/11/2026
12  | (ausente)      | Praca Central      | Em analise | 02/11/2026
13  | iluminacao

### Painel de indicadores

O painel mostra os dados de origem, as regras de cálculo e uma leitura curta dos resultados. Ele já inclui a ocorrência 13 cadastrada nas demonstrações.

In [15]:
ver_resumo_indicadores()

[ALERTA DE VALIDACAO] Ocorrencia ID 12 possui Categoria ausente/inconsistente!

      PAINEL DE INDICADORES (DASHBOARD)   
Total de ocorrencias cadastradas: 13
Registros com dados inconsistentes: 1

--- Indicadores por Situacao ---
Regra: proporcao = quantidade da situacao / total x 100
- Aberta: 5 (38.5%)
- Em analise: 5 (38.5%)
- Concluida: 3 (23.1%)

--- Indicadores por Categoria ---
Regra: proporcao = quantidade da categoria / total x 100
- iluminacao: 4 (30.8%)
- acessibilidade: 2 (15.4%)
- limpeza: 3 (23.1%)
- manutencao: 3 (23.1%)
- Indefinida: 1 (7.7%)

--- Matriz Sintese (Categoria x Situacao) ---
Categoria       | Aberta   | Em analise | Concluida
--------------------------------------------------
iluminacao      | 2        | 1          | 1        
acessibilidade  | 1        | 1          | 0        
limpeza         | 1        | 1          | 1        
manutencao      | 1        | 1          | 1        
Indefinida      | 0        | 1          | 0        

--- Indicadores Operac

## 9. Menu principal

`main()` repete o menu até o usuário digitar `0`. Entradas que não são números são recusadas antes de qualquer ação.

In [16]:
def mostrar_menu():
    print("\n--------- MENU PRINCIPAL ---------")
    print("1- Registrar uma Ocorrencia")
    print("2- Mostrar todas as ocorrencias")
    print("3- Atualizar situacao de uma ocorrencia")
    print("4- Encontrar ocorrencia pelo numero")
    print("5- Filtrar ocorrencias")
    print("6- Ver resumo das ocorrencias (Dashboard)")
    print("7- Ver historico recente (pilha)")
    print("8- Ver fila de analise (FIFO)")
    print("9- Ver hierarquia bairro > categoria")
    print("0- Sair")


def main():
    while True:
        mostrar_menu()
        opcao = input("\nEscolha uma opcao: ").strip()

        if not opcao.isdigit():
            print("Digite apenas numeros validos!")
            continue

        opcao = int(opcao)

        if opcao == 1:
            registrar_ocorrencia()
        elif opcao == 2:
            mostrar_todas_ocorrencias()
        elif opcao == 3:
            atualizar_situacao_ocorrencia()
        elif opcao == 4:
            encontrar_ocorrencia()
        elif opcao == 5:
            filtrar_ocorrencias()
        elif opcao == 6:
            ver_resumo_indicadores()
        elif opcao == 7:
            mostrar_historico()
        elif opcao == 8:
            mostrar_fila()
        elif opcao == 9:
            mostrar_hierarquia()
        elif opcao == 0:
            print("Obrigado por usar o Neighborhood Action Dashboard!")
            print("Ate logo!")
            break
        else:
            print("Opcao invalida! Escolha um numero do menu.")


## 10. Execução interativa

Execute a célula abaixo e use o campo de texto que aparece para navegar pelo menu. Digite `0` para sair.

**Sugestão de roteiro:** `1` (cadastrar) → `5` (filtrar categoria + status) → `3` (atualizar) → `7` (histórico) → `6` (dashboard).

In [ ]:
main()